In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

INPUT_FILE = "coverage_results_rich.json"

# loading coverage results
with open(INPUT_FILE, "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("\n✅ Loaded coverage results")
print(df.head())

# -------- MAIN COLUMN (t = 0.7) --------
col = "coverage_0.7"

# -------- SUMMARY STATISTICS --------
print("\n📊 SUMMARY STATISTICS (t=0.7)\n")

summary = {
    "Metric": ["Mean", "Median", "Std", "Min", "Max"],
    "Coverage (0.7)": [
        df[col].mean(),
        df[col].median(),
        df[col].std(),
        df[col].min(),
        df[col].max(),
    ]
}

summary_df = pd.DataFrame(summary)
print(summary_df)

# -------- HISTOGRAM --------
plt.figure()
plt.hist(df[col], bins=50)
plt.xlabel("Coverage")
plt.ylabel("Number of Datasets")
plt.title("Distribution of Coverage (t=0.7)")
plt.savefig("coverage_histogram.png")
plt.close()

#BOX PLOT
plt.boxplot(df[col])

median_val = df[col].median()

plt.text(
    1.05,
    median_val,
    f"Median: {median_val:.2f}",
    verticalalignment='center'
)

plt.ylabel("Coverage")
plt.title("Box Plot of Coverage (t=0.7)")
plt.savefig("coverage_boxplot.png", bbox_inches='tight')
plt.close()

# THRESHOLD CATEGORIES
def categorize(score):
    if score >= 0.75:
        return "High"
    elif score >= 0.5:
        return "Moderate"
    else:
        return "Low"

df["coverage_category"] = df[col].apply(categorize)

category_counts = df["coverage_category"].value_counts(normalize=True) * 100

print("\n📊 COVERAGE CATEGORY DISTRIBUTION (%)\n")
print(category_counts)

# -------- RANGE DISTRIBUTION (0–1) --------
print("\n📊 COVERAGE RANGE DISTRIBUTION (0–1)\n")

bins = [i/10 for i in range(11)]
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

df["coverage_bin"] = pd.cut(
    df[col],
    bins=bins,
    labels=labels,
    include_lowest=True
)

bin_counts = df["coverage_bin"].value_counts().sort_index()
bin_percent = (bin_counts / len(df)) * 100

range_df = pd.DataFrame({
    "Range": bin_counts.index,
    "Count": bin_counts.values,
    "Percentage": bin_percent.values
})

print(range_df)

# -------- RANGE BAR PLOT --------
plt.figure()
plt.bar(range_df["Range"], range_df["Percentage"])
plt.xticks(rotation=45)
plt.xlabel("Coverage Range")
plt.ylabel("Percentage of Datasets")
plt.title("Coverage Distribution (0–1 Range)")
plt.tight_layout()
plt.savefig("coverage_range_distribution.png")
plt.close()

# -------- MULTI-THRESHOLD COMPARISON --------
print("\n📊 MULTI-THRESHOLD COVERAGE COMPARISON\n")

threshold_means = df[["coverage_0.6", "coverage_0.7", "coverage_0.8"]].mean()

print(threshold_means)

# Plot threshold comparison
plt.figure()
plt.plot(["0.6", "0.7", "0.8"], threshold_means.values, marker='o')
plt.xlabel("Similarity Threshold")
plt.ylabel("Average Coverage")
plt.title("Coverage vs Similarity Threshold")
plt.savefig("coverage_threshold_comparison.png")
plt.close()

# -------- SCATTER: KEYWORD COUNT VS COVERAGE --------
plt.figure()
plt.scatter(df["num_keywords"], df[col])
plt.xlabel("Number of Keywords")
plt.ylabel("Coverage")
plt.title("Keyword Count vs Coverage")
plt.savefig("coverage_vs_keywords.png")
plt.close()

# -------- CORRELATION --------
print("\n🔗 CORRELATION MATRIX\n")

correlation = df[[
    "coverage_0.7",
    "avg_max_similarity",
    "num_keywords",
    "num_extracted"
]].corr()

print(correlation)

# -------- TOP & BOTTOM DATASETS --------
top = df.sort_values(col, ascending=False).head(10)
bottom = df.sort_values(col, ascending=True).head(10)

print("\n🏆 TOP 10 DATASETS (Coverage)\n")
print(top)

print("\n⚠️ BOTTOM 10 DATASETS (Coverage)\n")
print(bottom)

# -------- SAVE TABLES --------
summary_df.to_csv("coverage_summary.csv", index=False)
range_df.to_csv("coverage_range_distribution.csv", index=False)
category_counts.to_csv("coverage_category_distribution.csv")

print("\n💾 Files saved:")
print("- coverage_summary.csv")
print("- coverage_range_distribution.csv")
print("- coverage_category_distribution.csv")
print("- coverage_histogram.png")
print("- coverage_boxplot.png")
print("- coverage_range_distribution.png")
print("- coverage_threshold_comparison.png")
print("- coverage_vs_keywords.png")

print("\n✅ Coverage analysis complete!")


✅ Loaded coverage results
                                          dataset_id  num_keywords  \
0  http://data.europa.eu/88u/dataset/forestry-fac...             4   
1  http://data.europa.eu/88u/dataset/operational-...             6   
2  http://data.europa.eu/88u/dataset/2021-census-...             2   
3  http://data.europa.eu/88u/dataset/register-of-...             5   
4  http://data.europa.eu/88u/dataset/leaf-phenolo...             3   

   num_extracted  avg_max_similarity  coverage_0.6  coverage_0.7  coverage_0.8  
0              5            0.531037           0.4           0.2           0.0  
1              5            0.263706           0.0           0.0           0.0  
2              5            0.319779           0.0           0.0           0.0  
3              5            0.554658           0.4           0.4           0.4  
4              5            0.194141           0.0           0.0           0.0  

📊 SUMMARY STATISTICS (t=0.7)

   Metric  Coverage (0.7)
0    Mean